In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual para las gráficas
sns.set_theme(style="whitegrid")

# 1. Cargar los datos
# Asegúrate de que el archivo 'hotel_bookings.csv' esté en la misma carpeta
df = pd.read_csv('hotel-cancellation-prediction/data/raw/hotel_bookings.csv')

# 2. Vista previa rápida
print("--- TAMAÑO DEL DATASET ---")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print("\n--- PRIMERAS 5 FILAS ---")
display(df.head())

# 3. Diagnóstico de Tipos de Datos y Nulos
print("\n--- INFORMACIÓN DE COLUMNAS Y NULOS ---")
df.info()

# 4. Conteo exacto de valores nulos
print("\n--- VALORES FALTANTES POR COLUMNA ---")
nulos = df.isnull().sum()
print(nulos[nulos > 0].sort_values(ascending=False))

#5. Estadísticas descriptivas
df.describe(include='all').T

#6. Pruebas de valores anormales
huespedes_fantasma = (df['adults'] > 0) | (df['children'] > 0) | (df['babies'] > 0)
df_prueba = df[huespedes_fantasma].copy()
print(f"Filas eliminadas (reservas fantasma): {df.shape[0] - df_prueba.shape[0]}")
print(f"Total de filas finales: {df_prueba.shape[0]}")


--- TAMAÑO DEL DATASET ---
Filas: 119,390
Columnas: 32

--- PRIMERAS 5 FILAS ---


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03



--- INFORMACIÓN DE COLUMNAS Y NULOS ---
<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null 

Observamos que tenemos mas de 100,000 filas y 32 columnas, de las cuales tambien detectamos anomalias en el tipo de datos y valores nulos, ademas de encontrar valores fantasma (reservas sin ninguna persona).
Primero manejemos los valores nulos:
- Company tiene mas de 80% de datos faltantes, en esta variable lo normal seria el borrado de la columna, pero en nuestro caso utilizaremos la ausencia de esos datos como una caracteristica predictiva, conviertiendo la variable en binaria.
- Agent(16,340 nulos) Esta columna representa el ID de la agencia de vaijes, por lo tanto no hay errores, si no que el cliente hizo la reserva directamente con el hotel. Rellenaremos los nulos con 0.
- Country (488 nulos) Esta es una variable categrica la cual no podemos hacer una imputacion estadistica, lo mejor en este caso es rellenar con 'Unknown'.
- Children (4 nulos) Rellenaremos con la mediana natural (0)

En cuanto a la coreccion de Tipos convertiremos children y agent de tipos float a enteros ya que estos valores no pueden tener decimales.
Tambien las 180 reservas fantasma encontradas se eliminaran.


In [11]:
# Verificacion de los datos Limpios(El dataset proviene de la limpieza realizada en el notebook 01_hotel_eda.ipynb)
df_clean = pd.read_csv('hotel-cancellation-prediction/data/processed/hotel_bookings_clean_v1.csv')
print("--- VALORES FALTANTES DESPUÉS DE LA LIMPIEZA ---")
print(df_clean.isnull().sum().max()) 


--- VALORES FALTANTES DESPUÉS DE LA LIMPIEZA ---
0
